
# SMS Spam Classifier using DistilBERT

This notebook fine-tunes **DistilBERT** on the Hugging Face `sms_spam` dataset to classify messages as **ham** or **spam**.

All known issues are fixed:
- Correct text column name (`sms` instead of `text`)
- Safe train/eval split even if dataset is smaller than expected



## Step 1: Load & Explore Dataset


In [1]:

from datasets import load_dataset

dataset = load_dataset("sms_spam")
dataset

# Check column names
print("Columns:", dataset['train'].column_names)
print("Features:", dataset['train'].features)


C:\Users\Khizra\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Khizra\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Khizra\.cache\huggingface\hub\datasets--sms_spam. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrat

Columns: ['sms', 'label']
Features: {'sms': Value('string'), 'label': ClassLabel(names=['ham', 'spam'])}



## Step 2: Label Dictionaries


In [2]:

label_map = {0: 'ham', 1: 'spam'}
id_map = {'ham': 0, 'spam': 1}

label_map, id_map


({0: 'ham', 1: 'spam'}, {'ham': 0, 'spam': 1})


## Step 3: Tokenization & Preprocessing


In [3]:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

TEXT_COL = "sms"  # Correct column name for SMS text

def tokenize_fn(example):
    return tokenizer(
        example[TEXT_COL],
        padding='max_length',
        truncation=True,
        max_length=128
    )


C:\Users\Khizra\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Khizra\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:

tokenized_ds = dataset.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.remove_columns([TEXT_COL])
tokenized_ds = tokenized_ds.rename_column('label', 'labels')
tokenized_ds.set_format('torch')


Map: 100%|████████████████████████████████████████████████████████████████| 5574/5574 [00:01<00:00, 3300.54 examples/s]



## Step 4: Train / Evaluation Split


In [5]:

# Shuffle dataset
tokenized_ds = tokenized_ds.shuffle(seed=42)

total_size = len(tokenized_ds['train'])
train_size = min(5000, total_size)          # use 5000 or less if dataset is smaller
eval_size = total_size - train_size         # remaining samples

train_ds = tokenized_ds['train'].select(range(train_size))
eval_ds = tokenized_ds['train'].select(range(train_size, total_size))

print("Train size:", len(train_ds))
print("Eval size:", len(eval_ds))


Train size: 5000
Eval size: 574



## Step 5: Fine-Tune DistilBERT


In [6]:

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [7]:

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [8]:

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label=label_map,
    label2id=id_map
)


Loading weights: 100%|█| 100/100 [00:00<00:00, 158.38it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    save_strategy="no"
)
 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics
)



`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:

trainer.train()


C:\Users\Khizra\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.065245,0.984321,0.891566,1.000000,0.942675
2,0.055622,0.031963,0.991289,0.960000,0.972973,0.966443


C:\Users\Khizra\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=626, training_loss=0.04876854244512491, metrics={'train_runtime': 4318.2515, 'train_samples_per_second': 2.316, 'train_steps_per_second': 0.145, 'total_flos': 331168496640000.0, 'train_loss': 0.04876854244512491, 'epoch': 2.0})


## Step 6: Save Model


In [13]:

trainer.save_model("./spam_model")
tokenizer.save_pretrained("./spam_model")


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.54it/s]


('./spam_model\\tokenizer_config.json', './spam_model\\tokenizer.json')


## Step 7: Load Model & Predict


In [14]:

from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="./spam_model",
    tokenizer="./spam_model"
)

def predict_with_label(text):
    result = clf(text)[0]
    return f"{result['label']} (confidence: {result['score']:.2f})"


Loading weights: 100%|███████████████████| 104/104 [00:00<00:00, 217.48it/s, Materializing param=pre_classifier.weight]


In [15]:

texts = [
    "Congratulations! You've won a free ticket.",
    "Hey, are we meeting tomorrow?",
]

for t in texts:
    print(predict_with_label(t))


ham (confidence: 0.99)
ham (confidence: 1.00)
